# Validation Notebook

## Evaluation Metrics: Precision, Recall, F1

The `prf(pred, gold)` function computes **Precision**, **Recall**, and **F1** by comparing predicted values against a gold standard, row by row.

### Step 1 — Normalisation
Each value is lowercased and stripped; empty strings, `"nan"`, and `"none"` are mapped to a sentinel `"__none__"` (meaning "no extraction").

### Step 2 — True Positives
A prediction counts as a **True Positive (TP)** only when:
- the predicted value **matches** the gold value, **and**
- **both** are actual values (not `__none__`).

Rows where both pred and gold are `__none__` are **True Negatives** (correctly not extracted) and are excluded from TP.

### Step 3 — Denominators
| Count | Meaning | Equivalent |
|-------|---------|------------|
| `n_pred` | non-null predictions | TP + FP |
| `n_gold` | non-null gold values | TP + FN |

### Step 4 — Metrics

$$\text{Precision} = \frac{TP}{TP + FP} = \frac{TP}{n\_pred}$$

> *"Of everything the system extracted, how much is correct?"*

$$\text{Recall} = \frac{TP}{TP + FN} = \frac{TP}{n\_gold}$$

> *"Of everything that should have been found, how much did the system actually find?"*

$$F_1 = \frac{2 \cdot P \cdot R}{P + R}$$

> *Harmonic mean of Precision and Recall.*

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
import os
import sys
import pandas as pd

sys.path.insert(0, os.path.join(os.path.abspath('.'), 'Target-Event-Agent_Networks'))
from teanets.nlp_utils import get_spacy_nlp
from teanets.svo_extraction import extract_svos

nlp = get_spacy_nlp()

def norm(x):
    s = str(x).strip().lower()
    return "__none__" if s in {"", "nan", "none"} else s

def prf(pred, gold):
    p = pred.apply(norm)
    g = gold.apply(norm)
    both_present = (p != "__none__") & (g != "__none__")
    tp = ((p == g) & both_present).sum()
    n_pred = (p != "__none__").sum()
    n_gold = (g != "__none__").sum()
    precision = tp / n_pred if n_pred else 0.0
    recall = tp / n_gold if n_gold else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"TP": tp, "Pred": n_pred, "Gold": n_gold, "Precision": round(precision, 3), "Recall": round(recall, 3), "F1": round(f1, 3)}

[nltk_data] Downloading package wordnet to /home/seb/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [2]:
CORPUS_FILE = "../data/sample_sentences.csv"

df_corpus = pd.read_csv(CORPUS_FILE)
if "sentence" not in df_corpus.columns:
    raise ValueError("Corpus miss 'sentence' column.")

wdw_rows = []
for i, s in enumerate(df_corpus["sentence"].fillna("")):
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Agent") & (df_rel["TEA2"] == "Event")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Event") & (df_rel["TEA2"] == "Target")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    wdw_rows.append({
        "sent_id": i,
        "sentence": s,
        "agent": who,
        "event": did,
        "target": what,
    })

df_tea = pd.DataFrame(wdw_rows)
print(f"Corpus' sentences: {len(df_corpus)}")
print(f"Extracted Triplets (rows): {len(df_tea)}")
df_tea[["sentence", "agent", "event", "target"]].head(50)

Corpus' sentences: 122
Extracted Triplets (rows): 122


,sentence,agent,event,target
0,The cat chased the mouse.,cat,chase,mouse
1,John reads books.,john,read,book
2,Mary loves chocolate.,mary,love,chocolate
3,The dog barks loudly.,dog,bark loudly,NaN
4,The sun sets in the west.,sun,set,in west
5,Birds fly.,bird,fly,NaN
6,The student is reading a book.,student,read,book
7,They have finished their homework.,they,finish,their homework
8,He will eat the apple.,he,will eat,apple
9,The team had lost the game.,team,lose,game


In [3]:
GOLD_FILE = "../data/gold_standard_svo.csv"
df_gold = pd.read_csv(GOLD_FILE)

def extract_svo_dep(doc):
    """Baseline: pure dependency-based SVO, then map to semantic roles."""
    subj = verb = obj = None
    is_passive = False
    for token in doc:
        if token.dep_ == 'ROOT':
            verb = token.lemma_
            for child in token.children:
                if child.dep_ in ('nsubj', 'nsubjpass', 'csubj', 'csubjpass'):
                    subj = child.lemma_
                    if child.dep_ in ('nsubjpass', 'csubjpass'):
                        is_passive = True
                if child.dep_ in ('dobj', 'pobj', 'attr', 'acomp', 'ccomp'):
                    obj = child.lemma_
                elif child.dep_ == 'prep':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = f"{child.lemma_} {p_child.lemma_}"
                elif child.dep_ == 'agent':
                    for p_child in child.children:
                        if p_child.dep_ == 'pobj':
                            obj = p_child.lemma_
                            is_passive = True
                elif child.dep_ == 'xcomp':
                    for xc_child in child.children:
                        if xc_child.dep_ in ('dobj', 'pobj', 'attr'):
                            obj = xc_child.lemma_
            break
    # Map syntactic → semantic roles
    if is_passive and obj:
        # Passive with agent: subj→Target, obj→Agent
        return obj, verb, subj
    elif is_passive:
        # Passive without agent: subj→Agent (approx), no Target
        return subj, verb, None
    else:
        # Active: subj→Agent, obj→Target
        return subj, verb, obj

gold_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    doc = nlp(s)
    agent, event, target = extract_svo_dep(doc)
    gold_rows.append({
        "sent_id": row.get("sent_id", i),
        "sentence": row["sentence"],
        "pred_agent": agent,
        "pred_event": event,
        "pred_target": target,
        "gold_agent": row["agent"],
        "gold_event": row["event"],
        "gold_target": row["target"],
    })

df_eval = pd.DataFrame(gold_rows)
metrics_baseline = {
    "Agent":  prf(df_eval["pred_agent"],  df_eval["gold_agent"]),
    "Event":  prf(df_eval["pred_event"],  df_eval["gold_event"]),
    "Target": prf(df_eval["pred_target"], df_eval["gold_target"]),
}

# TEA Nets extraction
tea_rows = []
for i, row in df_gold.iterrows():
    s = str(row["sentence"])
    if s == 'nan': continue
    df_rel = extract_svos(nlp(s))

    who_did = df_rel[(df_rel["TEA"] == "Agent") & (df_rel["TEA2"] == "Event")] if not df_rel.empty else pd.DataFrame()
    did_what = df_rel[(df_rel["TEA"] == "Event") & (df_rel["TEA2"] == "Target")] if not df_rel.empty else pd.DataFrame()

    who = did = what = None
    if not who_did.empty:
        who = who_did.iloc[0]["Node 1"]
        did = who_did.iloc[0]["Node 2"]
    if not did_what.empty:
        if did is None:
            did = did_what.iloc[0]["Node 1"]
        same_verb = did_what[did_what["Node 1"] == did]
        what = same_verb.iloc[0]["Node 2"] if not same_verb.empty else did_what.iloc[0]["Node 2"]

    tea_rows.append({
        "pred_agent": who,
        "pred_event": did,
        "pred_target": what,
        "gold_agent": row["agent"],
        "gold_event": row["event"],
        "gold_target": row["target"],
    })

df_tea_eval = pd.DataFrame(tea_rows)
metrics_tea = {
    "Agent":  prf(df_tea_eval["pred_agent"],  df_tea_eval["gold_agent"]),
    "Event":  prf(df_tea_eval["pred_event"],  df_tea_eval["gold_event"]),
    "Target": prf(df_tea_eval["pred_target"], df_tea_eval["gold_target"]),
}

print("SVO Validation vs. Gold Standard (122 sentences, semantic roles):\n")
print("=== Baseline ===")
df_bl = pd.DataFrame(metrics_baseline).T
display(df_bl)

print("\n=== TEA Nets ===")
df_tea_m = pd.DataFrame(metrics_tea).T
display(df_tea_m)

SVO Validation vs. Gold Standard (122 sentences, semantic roles):

=== Baseline ===


,TP,Pred,Gold,Precision,Recall,F1
Agent,109.0,122.0,122.0,0.893,0.893,0.893
Event,85.0,122.0,122.0,0.697,0.697,0.697
Target,65.0,88.0,82.0,0.739,0.793,0.765



=== TEA Nets ===


,TP,Pred,Gold,Precision,Recall,F1
Agent,111.0,122.0,122.0,0.910,0.910,0.910
Event,99.0,122.0,122.0,0.811,0.811,0.811
Target,76.0,92.0,82.0,0.826,0.927,0.874


In [13]:
from teanets.svo_extraction import _passive_info

PASSIVE_GOLD = "../data/passive_gold.csv"
df_pgold = pd.read_csv(PASSIVE_GOLD)

pred_passive = []
for sent in df_pgold["sentence"].fillna(""):
    doc = nlp(str(sent))
    is_pass = False
    for token in doc:
        if token.pos_ in ("VERB", "AUX") and token.dep_ not in (
            "aux", "auxpass", "amod", "npadvmod", "prep",
        ):
            pinfo = _passive_info(token)
            if pinfo["is_passive"]:
                is_pass = True
                break
    pred_passive.append(int(is_pass))

df_pgold["pred_passive"] = pred_passive

tp = ((df_pgold["is_passive"] == 1) & (df_pgold["pred_passive"] == 1)).sum()
fp = ((df_pgold["is_passive"] == 0) & (df_pgold["pred_passive"] == 1)).sum()
fn = ((df_pgold["is_passive"] == 1) & (df_pgold["pred_passive"] == 0)).sum()
tn = ((df_pgold["is_passive"] == 0) & (df_pgold["pred_passive"] == 0)).sum()

# Per-class metrics
p_prec = tp / (tp + fp) if (tp + fp) else 0.0
p_rec  = tp / (tp + fn) if (tp + fn) else 0.0
p_f1   = 2 * p_prec * p_rec / (p_prec + p_rec) if (p_prec + p_rec) else 0.0

a_prec = tn / (tn + fn) if (tn + fn) else 0.0
a_rec  = tn / (tn + fp) if (tn + fp) else 0.0
a_f1   = 2 * a_prec * a_rec / (a_prec + a_rec) if (a_prec + a_rec) else 0.0

# Macro average
macro_prec = (p_prec + a_prec) / 2
macro_rec  = (p_rec + a_rec) / 2
macro_f1   = (p_f1 + a_f1) / 2


metrics_passive = {
    "Passive":  {"Precision": round(p_prec, 3), "Recall": round(p_rec, 3), "F1": round(p_f1, 3)},
    "Active":   {"Precision": round(a_prec, 3), "Recall": round(a_rec, 3), "F1": round(a_f1, 3)},
    "Macro Avg": {"Precision": round(macro_prec, 3), "Recall": round(macro_rec, 3), "F1": round(macro_f1, 3)},
}
df_passive_metrics = pd.DataFrame(metrics_passive).T
display(df_passive_metrics)

# Show mismatches
mismatches = df_pgold[df_pgold["is_passive"] != df_pgold["pred_passive"]]
if not mismatches.empty:
    print(f"\nMismatches ({len(mismatches)}):")
    for _, r in mismatches.iterrows():
        label = "FP" if r["pred_passive"] == 1 else "FN"
        print(f"  [{label}] {r['sentence'][:80]}")

,Precision,Recall,F1
Passive,0.900,0.954,0.926
Active,0.989,0.975,0.982
Macro Avg,0.944,0.965,0.954



Mismatches (33):
  [FN] An interesting election had occured.
  [FP] I am stunned at the impact politics is having on our country these days.
  [FN] Politics have become senseless.
  [FP] Getting involved in politics is easy.
  [FN] There were not enough votes to elect him.
  [FP] Politics can be stressful to be involved in. 
  [FN] I wish there had been different candidates running for President in the election
  [FN] Politics never interested me growing up.
  [FP] I think politics are overrated.
  [FN] The senator approval ratings had sunken since voting for the crime bill.
  [FN] Gymnastics is one of the sports I like watching. 
  [FP] Sports figures are paid exorbitant amounts of money.
  [FP] The field was filled with players.
  [FN] Sport are fun but are very difficult and time consuming when done professionally
  [FP] NBA fans are screwed
  [FP] The sports arena is filled.
  [FP] Many research articles are published.
  [FP] Science is known for its research.
  [FP] Science has c